# 02 — Data pipeline

Build the matched (harm, safe) pairs we'll patch.


In [ ]:
from safety_circuits.data import (
    load_advbench, load_harmbench, load_rtp, load_hh_harmless,
    build_matched_pairs, save_jsonl,
)
from safety_circuits.config import REPO_ROOT


In [ ]:
harm_adv = load_advbench(limit=256)
harm_hb  = load_harmbench(limit=128)
safe     = load_hh_harmless(limit=1024)
print(len(harm_adv), len(harm_hb), len(safe))


In [ ]:
pairs = build_matched_pairs(harm_adv + harm_hb, safe, n_pairs=128, seed=0)
out = REPO_ROOT / 'data' / 'processed' / 'pairs.jsonl'
# flatten to a list of dicts for jsonl
records = [
    {'harm': h.__dict__, 'safe': s.__dict__} for h, s in pairs
]
out.parent.mkdir(parents=True, exist_ok=True)
import json
with open(out, 'w') as f:
    for r in records: f.write(json.dumps(r) + '\n')
print('wrote', out, '—', len(records), 'pairs')


In [ ]:
import pandas as pd
df = pd.DataFrame({'harm_len': [len(h.text) for h, _ in pairs],
                   'safe_len': [len(s.text) for _, s in pairs]})
df.describe()
